# Lesson 02 - Utforske Microsoft Agent Framework

The **Microsoft Agent Framework (MAF)** er et samlet rammeverk for å bygge AI-agenter. Det gir en ren, komponérbar arkitektur med fire kjernebyggesteiner:

- **Client** – kobler til et AI-modellendepunkt og håndterer kommunikasjon
- **Agent** – pakker inn en klient med instruksjoner og verktøydefinisjoner
- **Tools** – utvider agentens muligheter med egendefinerte funksjoner modellen kan kalle
- **Session** – opprettholder samtalehistorikk for interaksjoner over flere runder

I denne leksjonen skal vi bygge en **reisebestillingsagent** som sjekker tilgjengeligheten til destinasjoner ved bruk av disse konseptene.


## Oppsett


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Forstå Agent-rammeverkets arkitektur

Microsoft Agent Framework følger en lagdelt arkitektur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – En `AzureAIProjectAgentProvider` kobler til en Azure OpenAI-distribusjon. Den håndterer autentisering, forespørselformatering og svardanalyse.
2. **Agent** – Opprettet fra klienten via `provider.create_agent()`, kombinerer agenten modelltilgang med instruksjoner (systemprompt) og verktøy.
3. **Tools** – Python-funksjoner dekorert med `@tool` som agenten kan påkalle for å utføre handlinger eller hente data.
4. **Session** – Et `AgentSession`-objekt (opprettet via `agent.create_session()`) som lagrer samtalehistorikk, og muliggjør samtaler med flere runder der agenten husker tidligere kontekst.

La oss bygge hvert lag steg for steg.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Legge til verktøy med @tool-dekoratøren

Verktøy lar agenter utføre handlinger utover å generere tekst. `@tool`-dekoratøren gjør en vanlig Python-funksjon om til noe agenten kan kalle.

Viktige punkter:
- Bruk `Annotated[type, "description"]` slik at modellen forstår hver parameter.
- Docstringen blir verktøybeskrivelsen modellen ser.
- `approval_mode="never_require"` betyr at verktøyet kjører automatisk uten brukerbekreftelse.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Lage en agent med verktøy

Nå kombinerer vi klienten, instruksjonene og verktøyene til en agent. `instructions` fungerer som systemprompten — de definerer agentens persona og oppførsel.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Flersporede Samtaler med Økter

En `AgentSession` (opprettet via `agent.create_session()`) holder oversikt over alle meldinger i en samtale. Ved å sende samme økt til hver `agent.run()`-anrop, har agenten tilgang til hele samtalehistorikken og kan referere tilbake til tidligere meldinger.

Vi sender `tools=[check_destination_availability]` slik at agenten kan bruke vår tilgjengelighetssjekker under hvert trinn.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Sammendrag

I denne leksjonen utforsket du de fire pilarene i Microsoft Agent-rammeverket:

| Konsept | Hva du lærte |
|---------|--------------|
| **Klient** | `AzureAIProjectAgentProvider` kobler til Azure OpenAI med autentisering basert på legitimasjon |
| **Agent** | `provider.create_agent()` pakker en modellkobling med instruksjoner og et navn |
| **Verktøy** | `@tool`-dekoratøren eksponerer Python-funksjoner for at agenten kan kalle dem |
| **Økt** | `agent.create_session()` opprettholder samtalehistorikk over flere runder |

Disse byggesteinene settes sammen for å lage agenter som kan holde naturlige samtaler, kalle eksterne funksjoner og opprettholde kontekst — grunnlaget for mer avanserte agentmønstre i senere leksjoner.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi tilstreber nøyaktighet, vennligst vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det originale dokumentet på det opprinnelige språket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
